# Course Timetabling Prototype

Bu notebook'u `model.py`, `validate.py` ve `data/` klasörüyle **aynı klasörde** açın.

**Veri sentetiktir**: Sabancı'ya ait değildir. Gerçek veri gelince sadece `data/` içindeki CSV'leri değiştirin.

Hücreleri yukarıdan aşağıya sırayla çalıştırın (Shift + Enter).

## 1. Paketleri kur (sadece ilk seferde)

In [ ]:
%pip install pulp highspy matplotlib pandas

## 2. Doğru klasörde miyiz?
Aşağıda `data` ve `model.py` görünmüyorsa notebook yanlış klasörde açılmış demektir.

In [ ]:
import os, glob
if not os.path.exists('model.py'):
    aday = []
    for yer in ['Downloads', 'Desktop', 'Documents', 'Music']:
        aday += glob.glob(os.path.expanduser(f'~/{yer}/**/timetabling_prototype/model.py'), recursive=True)
    if aday:
        os.chdir(os.path.dirname(aday[0]))
print('Klasör:', os.getcwd())
print('data/:', sorted(os.listdir('data')) if os.path.isdir('data') else 'BULUNAMADI - zip açılmamış olabilir')

## 3. Veriye bak
Kendi verinizi kullanacaksanız bu adımı atlayın; `generate_data.py` dosyaları baştan yazar.

In [ ]:
# Örnek veriyi yeniden üretmek için alttaki satırın başındaki # işaretini kaldırın
# %run generate_data.py

import pandas as pd
print('Sections:'); display(pd.read_csv('data/sections.csv'))
print('Rooms:'); display(pd.read_csv('data/rooms.csv'))

## Modelde hangi kısıtlar var?

**A)** Modelin içine yazılan kısıtlar ve kaçar tane oldukları. **B)** Çözümden önce uygulanan kurallar: bunları bozan saat–sınıf seçenekleri hiç oluşturulmuyor. Numaralar model dokümanındaki (1)–(5) ile aynı.

Kısıtların açık yazılmış birer örneğini görmek için ikinci hücreyi çalıştırın.

In [ ]:
%run show_constraints.py

In [ ]:
%run show_constraints.py --example

## 4. Modeli çöz — Seçenek A (yerleşmeyen dersleri minimize et)

In [ ]:
%run model.py --objective unplaced

In [ ]:
%run validate.py

In [ ]:
from IPython.display import Image
display(pd.read_csv('results/timetable.csv'))
Image('results/timetable_grid.png')

## 5. Seçenek B (boş koltukları minimize et)
Aynı veride farklı amaçla nasıl farklı bir program çıktığını karşılaştırın.

In [ ]:
%run model.py --objective seats

In [ ]:
%run validate.py
display(pd.read_csv('results/room_utilization.csv'))
Image('results/timetable_grid.png')

## 6. Neden yerleşmedi? Hangi kural gevşerse yerleşir?

`scenario_tight` aynı verinin **bilerek zorlaştırılmış** hali: 120 kişilik tek sınıf kapalı (tadilat gibi) ve INS06 sadece Pazartesi–Salı sabahları müsait.

Betik önce normal çözüyor ve yerleşemeyenleri listeliyor. Sonra **hepsini yerleştirmek için en az hangi kuralların bozulması gerektiğini** buluyor.

Hiç gevşetilmeyen kurallar: aynı sınıfta aynı anda iki ders, bir hocanın iki yerde olması, gerekli donanım.

In [ ]:
%run diagnose.py --data scenario_tight

In [ ]:
display(pd.read_csv('results/diagnosis.csv'))

### Kural ağırlıklarını değiştirmek

Hangi kuralın gevşetilmesinin daha "pahalı" olduğu `scenario_tight/relaxation_weights.csv` dosyasında. Örneğin hoca müsaitliğini neredeyse hiç bozulmayacak hale getirmek için R5'in ağırlığını 10 yapıp betiği tekrar çalıştırın.

**Not:** Ağırlıklar şu an yer tutucu (hepsi eşit). Hangi kuralın daha kolay esnetilebileceğine Öğrenci İşleri karar vermeli.

In [ ]:
display(pd.read_csv('scenario_tight/relaxation_weights.csv'))

## 7. Gerçek ölçekte örnek veri (scenario_full)

İrem Hanım'ın verdiği sayılarla: **654 ders, 1.599 section, 114 sınıf**; zaman ızgarası 5 gün × 12 saat (varsayım). Bu üç sayı dışındaki her şey **uydurma** (sınıf boyutları, hocalar, kayıt sayıları, bloklar).

Bu veride model **çözülmüyor**, sadece ne kadar büyük olduğu ölçülüyor. Tek aşamalı model bilgisayarda kurulamayacak kadar büyük; `model.py` bu yüzden 1 milyon değişkenin üstünde kendini durduruyor.

In [ ]:
# Veriyi tekrar üretmek isterseniz (aynı sonucu verir):
# %run generate_full_data.py

print('Sections:', len(pd.read_csv('scenario_full/sections.csv')))
print('Rooms   :', len(pd.read_csv('scenario_full/rooms.csv')))
display(pd.read_csv('scenario_full/sections.csv').head(10))

In [ ]:
%run model_size.py --data scenario_full